# 05 — Entraînement dans les conditions de production

**Projet** : Prédiction d'attrition client (churn télécom) avec LightGBM
**Ce que fait `make train`** : charger → valider → splitter → features → preprocessing →
entraîner → métriques de validation → **persister** (modèle, fiche, métriques, splits).

Ce notebook exécute exactement le même chemin, mais de façon instrumentée : chaque brique est
visible, mesurable et rejouable.

## Objectifs pédagogiques

1. Piloter un entraînement par **objets** (Trainer, callbacks) plutôt que par un script monolithique.
1. Lire un `TrainingOutcome` : métriques, durée, historique, artefacts.
1. Mesurer la **stabilité** d'un modèle (plusieurs graines) avant de conclure.
1. Vérifier le garde-fou de qualité déclaré en configuration.

**Objectifs transverses du dépôt**

- Comprendre la croissance leaf-wise (`num_leaves`) et son risque de sur-apprentissage face à `max_depth`.
- Armer l'early stopping natif via un callback LightGBM (`early_stopping`, `log_evaluation`) sans réécrire la boucle.
- Régulariser un booster (min_child_samples, subsample + subsample_freq, colsample_bytree, reg_lambda, min_split_gain).

In [1]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.7)")
print(f"Cible             : {CONFIG.data.target}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

Projet            : telecom-churn-lightgbm
Tâche             : binary
Métrique primaire : roc_auc (seuil cible : 0.7)
Cible             : churned
Algorithme        : lightgbm (Gradient boosting LightGBM)
Lignes (notebook) : 1500


In [2]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

2026-09-13 06:04:24 | INFO     | src.data.loaders:_log_profile:183 - Loaded raw data | rows=4000 cols=15 missing_cells=317 memory=631.8 KB


Dataset lu depuis data/raw/telecom_churn.parquet
shape = (1500, 15)


,customer_id,signup_date,tenure_months,contract_type,internet_service,payment_method,region,monthly_charges,total_charges,support_tickets_6m,avg_monthly_data_gb,num_products,has_promotion,satisfaction_score,churned
0,CUS-00001,2024-12-21,13,two_year,dsl,electronic_check,south,51.89,680.43,1,83.89,2,0,7.57,0
1,CUS-00002,2022-05-03,45,two_year,dsl,credit_card,south,40.44,1849.09,0,138.90,3,0,7.41,0
2,CUS-00003,2023-07-21,30,two_year,none,mailed_check,west,20.11,606.84,1,3.18,5,0,7.48,0
3,CUS-00004,2025-12-01,2,month_to_month,dsl,credit_card,west,49.56,100.25,0,54.37,3,1,5.78,1
4,CUS-00005,2025-08-12,5,month_to_month,fiber,electronic_check,west,84.84,406.24,1,161.71,3,0,5.67,1


In [3]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

2026-09-13 06:04:24 | INFO     | src.data.loaders:split:505 - Split (random) | train=975 val=225 test=300 | stratify=True


2026-09-13 06:04:24 | INFO     | src.features.build_features:fit:257 - FeatureBuilder fitted | recipes=7 learned=['charges_by_contract', 'tenure_bucket']


2026-09-13 06:04:24 | INFO     | src.preprocessing.pipelines:fit:318 - Fitting preprocessing | rows=975 numeric=17 categorical=4 encoder=onehot scaler=standard


2026-09-13 06:04:24 | INFO     | src.preprocessing.pipelines:fit:332 - Preprocessing fitted | output_features=31


train : (975, 31)
val   : (225, 31)
test  : (300, 31)
features livrées au modèle : 31


,tenure_months,monthly_charges,total_charges,support_tickets_6m,avg_monthly_data_gb,num_products,has_promotion,satisfaction_score,charges_per_tenure,tickets_per_product,tenure_bucket,heavy_data_user,low_satisfaction,charges_by_contract,signup_year,signup_month,signup_quarter,contract_type_month_to_month,contract_type_one_year,contract_type_two_year,internet_service_dsl,internet_service_fiber,internet_service_none,payment_method_bank_transfer,payment_method_credit_card,payment_method_electronic_check,payment_method_mailed_check,region_east,region_north,region_south,region_west
0,-0.454667,-0.376656,-0.100429,-0.936103,-0.677167,1.815339,-0.554049,0.352466,-0.338491,-0.862593,-0.385793,-0.4381,-0.111629,-0.909537,0.226664,1.318529,1.347130,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.947387,-1.777863,-2.166664,-0.936103,-1.189225,-0.636123,1.804894,0.676763,-0.187523,-0.862593,-1.262595,-0.4381,-0.111629,-0.878735,0.845706,0.737351,0.456654,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,-0.898115,0.669337,-0.794432,-0.036004,-0.161837,-0.636123,-0.554049,0.268389,0.997783,0.066669,-1.262595,-0.4381,-0.111629,-0.909537,0.845706,0.446762,0.456654,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-0.602483,-1.920453,-1.320154,-0.936103,-1.184157,-0.636123,-0.554049,-0.464283,-0.617402,-0.862593,-0.385793,-0.4381,-0.111629,-0.878735,0.845706,-1.296772,-1.324298,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.185869,-0.129265,0.611829,-0.936103,-0.739353,0.998185,-0.554049,0.760840,-0.552362,-0.862593,0.491009,-0.4381,-0.111629,-0.878735,-0.392377,1.027940,1.347130,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


## 1. Le modèle, construit depuis la configuration

In [4]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)
print(MODEL.summary())
pd.Series(FIT_RESULT.metrics, name="métrique").to_frame("valeur")

2026-09-13 06:04:24 | INFO     | src.models.model:_build_estimator:249 - Estimator LGBMClassifier | task=binary params={'boosting_type': 'gbdt', 'subsample_freq': 1, 'n_estimators': 500, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.06, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.0, 'min_split_gain': 0.0, 'importance_type': 'gain', 'verbosity': -1, 'random_state': 42, 'n_jobs': -1}


2026-09-13 06:04:24 | INFO     | src.models.factory:build_model:127 - Model built | LightGBMModel(algorithm=lightgbm, boosting=gbdt, rounds=500, num_leaves=31, task=binary, features=31, state=not fitted)


2026-09-13 06:04:24 | INFO     | src.models.model:_build_estimator:249 - Estimator LGBMClassifier | task=binary params={'boosting_type': 'gbdt', 'subsample_freq': 1, 'n_estimators': 500, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.06, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.0, 'min_split_gain': 0.0, 'importance_type': 'gain', 'verbosity': -1, 'random_state': 42, 'n_jobs': -1}


2026-09-13 06:04:24 | INFO     | src.models.model:_apply_class_weight:478 - scale_pos_weight='auto' résolu à 2.7938 (négatifs/positifs)


2026-09-13 06:04:25 | INFO     | src.models.model:fit:404 - Model fitted in 0.08s | {'fit_seconds': 0.07590624699992077, 'n_boosting_rounds': 500.0, 'best_iteration': 59.0, 'best_validation_score': 0.8736981825607515}


LightGBMModel(algorithm=lightgbm, boosting=gbdt, rounds=500, num_leaves=31, task=binary, features=31, state=fitted)


,valeur
fit_seconds,0.075906
n_boosting_rounds,500.000000
best_iteration,59.000000
best_validation_score,0.873698


**Ce qu'il faut retenir**

- `build_model` est la **seule** porte d'entrée : le reste du projet ne connaît que `BaseModel`.
- Changer de stack (XGBoost, PyTorch, …) ne modifie ni le trainer, ni l'évaluateur, ni l'inférence.
- Les hyperparamètres viennent de `conf/model/default.yaml` : aucun n'est codé en dur ici.

## 2. Callbacks : instrumenter sans polluer la boucle d'entraînement

In [5]:
from src.training.callbacks import (
    EarlyStoppingCallback,
    LoggingCallback,
    MetricHistoryCallback,
    MetricThresholdCallback,
)
from src.training.trainer import Trainer, TrainingData

TRAINING_DATA = TrainingData(
    X_train=PREPARED["X_train"],
    y_train=PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    feature_names=PREPARED["feature_names"],
    task=CONFIG.metrics.task,
)

CALLBACKS = [
    LoggingCallback(every=1),
    MetricHistoryCallback(),
    EarlyStoppingCallback(
        monitor=CONFIG.train.early_stopping.monitor,
        patience=CONFIG.train.early_stopping.patience,
        mode=CONFIG.train.early_stopping.mode,
    ),
    MetricThresholdCallback(
        monitor=f"val_{CONFIG.metrics.primary}",
        threshold=float(CONFIG.metrics.min_primary or 0.0),
        mode="max" if CONFIG.metrics.direction == "maximize" else "min",
    ),
]
[callback.name for callback in CALLBACKS]

['logging', 'metric-history', 'early-stopping', 'metric-threshold']

In [6]:
TRAINER = Trainer(
    MODEL,
    config=CONFIG.model_dump(),
    paths=NB_PATHS,
    metric_names=CONFIG.metrics.all_metrics,
    task=CONFIG.metrics.task,
    callbacks=CALLBACKS,
)
OUTCOME = TRAINER.train(TRAINING_DATA)

metrics_frame = pd.DataFrame(
    {
        "métrique": list(OUTCOME.metrics),
        "valeur": [OUTCOME.metrics[name] for name in OUTCOME.metrics],
    }
).sort_values("valeur", ascending=False)
print(f"durée : {OUTCOME.duration_seconds:.2f}s | artefacts : {len(OUTCOME.artifacts)}")
metrics_frame.round(4).reset_index(drop=True)

2026-09-13 06:04:25 | INFO     | src.training.trainer:train:221 - Trainer started | model=LightGBMModel(algorithm=lightgbm, boosting=gbdt, rounds=500, num_leaves=31, task=binary, features=31, state=fitted) | {'n_train': 975, 'n_val': 225, 'n_features': 31, 'task': 'binary'}


2026-09-13 06:04:25 | INFO     | src.models.model:_build_estimator:249 - Estimator LGBMClassifier | task=binary params={'boosting_type': 'gbdt', 'subsample_freq': 1, 'n_estimators': 500, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.06, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.0, 'min_split_gain': 0.0, 'importance_type': 'gain', 'verbosity': -1, 'random_state': 42, 'n_jobs': -1}


2026-09-13 06:04:25 | INFO     | src.training.callbacks:on_train_begin:119 - Training started | model=LightGBMModel | epochs=500 | params={'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.8, 'importance_type': 'gain', 'learning_rate': 0.06, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 1.5, 'subsample': 0.9, 'subsample_for_bin': 200000, 'subsample_freq': 1, 'verbosity': -1}


2026-09-13 06:04:25 | INFO     | src.models.model:_apply_class_weight:478 - scale_pos_weight='auto' résolu à 2.7938 (négatifs/positifs)


2026-09-13 06:04:25 | INFO     | src.training.callbacks:on_epoch_end:135 - Epoch 60/500 | best_iteration=59.00000 | best_validation_score=0.87370 | fit_seconds=0.06181 | n_boosting_rounds=500.00000


...

2026-09-13 06:04:25 | INFO     | src.models.model:save:655 - Model saved | /home/user/templates/data-science/classification/with-lightgbm/outputs/notebooks/artifacts/models/model.joblib (82277 bytes)


2026-09-13 06:04:25 | INFO     | src.training.trainer:save_artifacts:362 - Artefacts written: {'model': '/home/user/templates/data-science/classification/with-lightgbm/outputs/notebooks/artifacts/models/model.joblib', 'model_card': '/home/user/templates/data-science/classification/with-lightgbm/outputs/notebooks/artifacts/models/model_card.json', 'metrics': '/home/user/templates/data-science/classification/with-lightgbm/outputs/notebooks/artifacts/metrics/training_metrics.json'}


2026-09-13 06:04:25 | INFO     | src.training.trainer:train:258 - Trainer finished in 0.07s | fit_seconds=0.06181, n_boosting_rounds=500.00000, best_iteration=59.00000, best_validation_score=0.87370, val_roc_auc=0.87370, val_pr_auc=0.73946


durée : 0.07s | artefacts : 3


,métrique,valeur
0,n_boosting_rounds,500.0000
1,best_iteration,59.0000
2,best_validation_score,0.8737
3,val_roc_auc,0.8737
4,val_accuracy,0.8133
5,val_balanced_accuracy,0.7916
6,val_recall,0.7458
7,val_pr_auc,0.7395
8,val_f1,0.6769
9,val_precision,0.6197


**Ce qu'il faut retenir**

- Les métriques préfixées `val_` viennent du **split de validation** : elles ne sont jamais calculées sur le test.
- Le test reste vierge jusqu'au notebook 06 — c'est la condition d'une estimation honnête.
- La durée est tracée : un entraînement qui double soudainement signale une dérive de données ou de configuration.

## 3. Artefacts produits

In [7]:
import json

artifacts = pd.DataFrame(
    {
        "artefact": list(OUTCOME.artifacts),
        "chemin": [OUTCOME.artifacts[name] for name in OUTCOME.artifacts],
    }
)
display(artifacts)

card_path = OUTCOME.artifacts.get("model_card")
if card_path:
    card = json.loads(Path(card_path).read_text(encoding="utf-8"))
    print("fiche modèle — clés :", sorted(card))
    print("features attendues  :", len(card["feature_names"]))
    print("versions librairies :", card["library_versions"])

,artefact,chemin
0,model,/home/user/templates/data-science/classificati...
1,model_card,/home/user/templates/data-science/classificati...
2,metrics,/home/user/templates/data-science/classificati...


fiche modèle — clés : ['algorithm', 'artifacts', 'created_at', 'extra', 'feature_names', 'framework', 'library_versions', 'metrics', 'model_name', 'n_features', 'n_samples', 'notes', 'params', 'target_name', 'task']
features attendues  : 31
versions librairies : {'python': '3.11.2', 'platform': 'Linux-6.1.158+-x86_64-with-glibc2.36', 'numpy': '2.4.6', 'pandas': '3.0.5', 'pyarrow': '25.0.1', 'scipy': '1.17.1', 'sklearn': '1.9.1', 'pandera': '0.33.1', 'pydantic': '2.13.5', 'hydra': '1.3.6', 'xgboost': '3.2.0', 'lightgbm': '4.7.0'}


**Ce qu'il faut retenir**

- La **fiche modèle** (features, paramètres, versions, métriques) rend un artefact reproductible et auditable.
- Sans elle, un `.joblib` retrouvé dans six mois est inutilisable : impossible de savoir quoi lui donner en entrée.
- Les artefacts du notebook sont écrits dans `outputs/notebooks` ; `make train` écrit dans `artifacts/`.

## 4. Stabilité : plusieurs graines, même conclusion ?

In [8]:
from src.training.losses_metrics import MetricCalculator, MetricInputs

scores = []
for seed in (7, 21, 42):
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"])
    candidate.random_state = seed
    _ = candidate.fit(
        PREPARED["X_train"],
        PREPARED["y_train"],
        X_val=PREPARED["X_val"],
        y_val=PREPARED["y_val"],
        callbacks=[],
    )
    values = MetricCalculator(task=CONFIG.metrics.task, metrics=[CONFIG.metrics.primary]).evaluate(
        MetricInputs(
            y_true=PREPARED["y_val"],
            y_pred=candidate.predict(PREPARED["X_val"]),
            y_proba=candidate.predict_proba(PREPARED["X_val"])
            if candidate.supports_proba
            else None,
        )
    )
    scores.append(values.get(CONFIG.metrics.primary, float("nan")))

scores_array = np.asarray(scores, dtype="float64")
stability = pd.DataFrame(
    {
        "graine": [7, 21, 42],
        CONFIG.metrics.primary: scores_array.round(4),
    }
)
mean = float(np.nanmean(scores_array))
std = float(np.nanstd(scores_array, ddof=1)) if len(scores_array) > 1 else 0.0
print(f"moyenne = {mean:.4f} | ecart-type = {std:.4f}")
print(f"intervalle +/- 1 ecart-type = [{mean - std:.4f} ; {mean + std:.4f}]")
stability

2026-09-13 06:04:25 | INFO     | src.models.model:_build_estimator:249 - Estimator LGBMClassifier | task=binary params={'boosting_type': 'gbdt', 'subsample_freq': 1, 'n_estimators': 500, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.06, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.0, 'min_split_gain': 0.0, 'importance_type': 'gain', 'verbosity': -1, 'random_state': 42, 'n_jobs': -1}


2026-09-13 06:04:25 | INFO     | src.models.factory:build_model:127 - Model built | LightGBMModel(algorithm=lightgbm, boosting=gbdt, rounds=500, num_leaves=31, task=binary, features=31, state=not fitted)


2026-09-13 06:04:25 | INFO     | src.models.model:_build_estimator:249 - Estimator LGBMClassifier | task=binary params={'boosting_type': 'gbdt', 'subsample_freq': 1, 'n_estimators': 500, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.06, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.0, 'min_split_gain': 0.0, 'importance_type': 'gain', 'verbosity': -1, 'random_state': 7, 'n_jobs': -1}


2026-09-13 06:04:25 | INFO     | src.models.model:_apply_class_weight:478 - scale_pos_weight='auto' résolu à 2.7938 (négatifs/positifs)


2026-09-13 06:04:25 | INFO     | src.models.model:fit:404 - Model fitted in 0.04s | {'fit_seconds': 0.041717011999935494, 'n_boosting_rounds': 500.0, 'best_iteration': 23.0, 'best_validation_score': 0.8708392893608332}


...

2026-09-13 06:04:25 | INFO     | src.models.model:_build_estimator:249 - Estimator LGBMClassifier | task=binary params={'boosting_type': 'gbdt', 'subsample_freq': 1, 'n_estimators': 500, 'num_leaves': 31, 'max_depth': -1, 'learning_rate': 0.06, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.0, 'min_split_gain': 0.0, 'importance_type': 'gain', 'verbosity': -1, 'random_state': 42, 'n_jobs': -1}


2026-09-13 06:04:25 | INFO     | src.models.model:_apply_class_weight:478 - scale_pos_weight='auto' résolu à 2.7938 (négatifs/positifs)


2026-09-13 06:04:25 | INFO     | src.models.model:fit:404 - Model fitted in 0.07s | {'fit_seconds': 0.07043960600003629, 'n_boosting_rounds': 500.0, 'best_iteration': 59.0, 'best_validation_score': 0.8736981825607515}


moyenne = 0.8722 | ecart-type = 0.0014
intervalle +/- 1 ecart-type = [0.8707 ; 0.8736]


,graine,roc_auc
0,7,0.8708
1,21,0.8720
2,42,0.8737


**Ce qu'il faut retenir**

- Un écart-type élevé signifie que le résultat dépend de la graine : toute comparaison de modèles doit le mesurer.
- Règle pratique : ne pas célébrer un gain inférieur à 2 × l'écart-type observé.
- Cette variabilité est aussi un argument pour la **validation croisée** en CI (`train.cross_validation`).

## 5. Garde-fou de qualité

In [9]:
threshold_callback = next(
    (callback for callback in CALLBACKS if isinstance(callback, MetricThresholdCallback)), None
)
threshold = CONFIG.metrics.min_primary
observed = OUTCOME.metrics.get(f"val_{CONFIG.metrics.primary}", float("nan"))
verdict = (
    "INCONNU" if not np.isfinite(observed) else ("OK" if observed >= float(threshold) else "ÉCHEC")
)
print(f"métrique          : val_{CONFIG.metrics.primary}")
print(f"valeur observée   : {observed:.4f}")
print(f"seuil configuré   : {threshold}")
print(f"verdict           : {verdict}")
print(f"callback satisfait: {getattr(threshold_callback, 'satisfied', 'n/a')}")

métrique          : val_roc_auc
valeur observée   : 0.8737
seuil configuré   : 0.7
verdict           : OK
callback satisfait: False


**Ce qu'il faut retenir**

- Le seuil vit dans `conf/config.yaml` (`metrics.min_primary`) : la CI l'utilise comme gate de déploiement.
- Un modèle sous le seuil ne doit **pas** être promu — même s'il « marche » en apparence.

## 6. Rechargement et vérification

In [10]:
from src.models import load_model

RESTORED = load_model(OUTCOME.artifacts["model"])
original = np.asarray(MODEL.predict(PREPARED["X_test"]), dtype="float64")
reloaded = np.asarray(RESTORED.predict(PREPARED["X_test"]), dtype="float64")
print("modèle rechargé :", RESTORED.summary())
print("prédictions identiques :", bool(np.allclose(original, reloaded)))

2026-09-13 06:04:25 | INFO     | src.models.factory:load_model:153 - Model loaded | model.joblib | LightGBMModel(algorithm=lightgbm, boosting=gbdt, rounds=500, num_leaves=31, task=binary, features=31, state=fitted)


modèle rechargé : LightGBMModel(algorithm=lightgbm, boosting=gbdt, rounds=500, num_leaves=31, task=binary, features=31, state=fitted)
prédictions identiques : True


**Ce qu'il faut retenir**

- Le test de rechargement est **le** test de déploiement : un modèle qui ne se recharge pas ne se sert pas.
- Il est rejoué automatiquement dans `tests/test_models.py::TestPersistence`.

## Synthèse

| Étape | Objet utilisé | Artefact |
| --- | --- | --- |
| Construction | `build_model(CONFIG)` | — |
| Entraînement | `Trainer.train(TrainingData)` | `model.joblib` |
| Instrumentation | `LoggingCallback`, `MetricHistoryCallback`, `EarlyStoppingCallback`, `MetricThresholdCallback` | historique |
| Traçabilité | `ModelCard` | `model_card.json` |
| Métriques | `MetricCalculator` | `training_metrics.json` |
| Qualité | `metrics.min_primary` | verdict CI |

**Suite** : `06_error_analysis.ipynb` évalue sur le **test** et transforme les erreurs en décisions.